## 🎯 Learning Objectives
* Understand the importance of evaluating retrieval quality in RAG systems.
* Define and calculate key retrieval metrics: Recall, Mean Reciprocal Rank (MRR), and Normalized Discounted Cumulative Gain (NDCG).
* Interpret the results of these metrics and understand their implications for RAG system performance.
* Identify appropriate use cases for each retrieval metric in different RAG scenarios.


## Evaluating Retrieval Quality: Recall, MRR, and NDCG

In a Retrieval-Augmented Generation (RAG) system, the 'Retrieval' component is the bedrock of its performance. If the retriever fails to fetch relevant information, even the most sophisticated Large Language Model (LLM) will struggle to generate accurate and helpful responses. Therefore, rigorously evaluating the quality of retrieval is paramount. This lesson introduces three fundamental metrics for assessing retrieval performance: Recall, Mean Reciprocal Rank (MRR), and Normalized Discounted Cumulative Gain (NDCG).

Imagine you're a librarian trying to find books for a patron's specific research topic. You go to the shelves, pull out a few books, and present them. How do you know if you did a good job?

### 1. Recall: Did we find *all* the relevant books?

Recall measures the proportion of truly relevant documents that were successfully retrieved by the system. It answers the question: "Out of all the relevant items that exist, how many did our system actually find?"

*   **Analogy**: If there are 10 perfectly relevant books for the patron's topic, and you only found 7 of them, your recall is 70%. You missed 3. High recall is crucial when you want to ensure that as much relevant information as possible is presented to the LLM, minimizing the chance of missing critical facts.
*   **Formula**: `Recall = (Number of relevant documents retrieved) / (Total number of relevant documents in the corpus)`
*   **Recall@k**: Often, we evaluate recall within the top `k` retrieved documents, as the LLM typically only processes a limited context window.

### 2. Mean Reciprocal Rank (MRR): How quickly did we find the *first* relevant book?

MRR is particularly useful when the order of results matters significantly, and finding *any* relevant item quickly is a high priority. It's commonly used in search engine evaluations.

*   **Analogy**: The patron just needs *one* good book to get started. If the first book you hand them is relevant, that's excellent! If they have to look through 5 books before finding a relevant one, that's less ideal. MRR assigns a score based on the rank of the *first* relevant item. If the first relevant item is at rank 1, the score is 1. If it's at rank 3, the score is 1/3. If no relevant item is found, the score is 0. The MRR is the average of these scores across multiple queries.
*   **Formula**: For a single query, `Reciprocal Rank = 1 / (Rank of the first relevant document)`. `MRR = (Sum of Reciprocal Ranks for all queries) / (Number of queries)`

### 3. Normalized Discounted Cumulative Gain (NDCG): How good are the top-ranked books, considering their relevance and position?

NDCG is a more sophisticated metric that accounts for both the relevance of retrieved documents and their position in the ranked list. It's ideal when documents can have varying degrees of relevance (e.g., 'highly relevant', 'moderately relevant', 'slightly relevant') and when placing more relevant items higher in the list is desirable.

*   **Analogy**: You hand the patron a stack of books. Some are perfect (relevance 3), some are good (relevance 2), some are okay (relevance 1), and some are irrelevant (relevance 0). NDCG rewards you more for putting the 'perfect' books at the top of the stack and penalizes you for placing less relevant books higher up. It then normalizes this score against the 'ideal' ranking (where all the most relevant books are at the very top).
*   **Core Concepts**:
    *   **Gain (G)**: The relevance score of a document.
    *   **Discounted Cumulative Gain (DCG)**: Sum of gains, where each gain is discounted by its position. Higher positions get less discount. `DCG = sum(rel_i / log2(i+1))` where `rel_i` is the relevance of the document at position `i` (1-indexed).
    *   **Ideal Discounted Cumulative Gain (IDCG)**: The maximum possible DCG, achieved by ranking all relevant documents in decreasing order of their relevance.
    *   **NDCG**: `DCG / IDCG`. This normalizes the score between 0 and 1, making it comparable across different queries.
*   **NDCG@k**: Like Recall, NDCG is often calculated for the top `k` results.

These metrics provide a comprehensive view of your RAG system's retrieval performance, allowing you to fine-tune your retriever, compare different retrieval strategies, and ultimately build a more effective RAG application.


In [ ]:
import numpy as np

def calculate_recall_at_k(ground_truth_relevant_docs, retrieved_docs, k):
    """
    Calculates Recall@k for a single query.

    Args:
        ground_truth_relevant_docs (set): A set of document IDs known to be relevant.
        retrieved_docs (list): A list of document IDs retrieved by the system, sorted by relevance score.
        k (int): The number of top documents to consider.

    Returns:
        float: The Recall@k score.
    """
    if not ground_truth_relevant_docs:
        return 1.0 # If there are no relevant documents, perfect recall is achieved vacuously.
    
    retrieved_at_k = set(retrieved_docs[:k])
    relevant_retrieved = len(ground_truth_relevant_docs.intersection(retrieved_at_k))
    return relevant_retrieved / len(ground_truth_relevant_docs)


def calculate_mrr(ground_truth_relevant_docs, retrieved_docs):
    """
    Calculates Mean Reciprocal Rank (MRR) for a single query.

    Args:
        ground_truth_relevant_docs (set): A set of document IDs known to be relevant.
        retrieved_docs (list): A list of document IDs retrieved by the system, sorted by relevance score.

    Returns:
        float: The Reciprocal Rank score for the query.
    """
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id in ground_truth_relevant_docs:
            return 1.0 / (i + 1) # Rank is 1-indexed
    return 0.0 # No relevant document found


def calculate_ndcg_at_k(ground_truth_relevant_docs, retrieved_docs, k):
    """
    Calculates Normalized Discounted Cumulative Gain (NDCG@k) for a single query.
    Assumes binary relevance (1 if in ground_truth_relevant_docs, 0 otherwise).

    Args:
        ground_truth_relevant_docs (set): A set of document IDs known to be relevant.
        retrieved_docs (list): A list of document IDs retrieved by the system, sorted by relevance score.
        k (int): The number of top documents to consider.

    Returns:
        float: The NDCG@k score.
    """
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_docs[:k]):
        relevance = 1 if doc_id in ground_truth_relevant_docs else 0
        dcg += relevance / np.log2(i + 2) # Position is i+1, so log2(i+1+1) = log2(i+2)

    # Calculate Ideal DCG (IDCG)
    # Sort relevant documents by their ideal relevance (all 1s for binary relevance)
    ideal_relevance_scores = sorted([1 for doc_id in ground_truth_relevant_docs], reverse=True)
    idcg = 0.0
    for i, relevance in enumerate(ideal_relevance_scores[:k]):
        idcg += relevance / np.log2(i + 2)

    if idcg == 0:
        return 0.0 # No relevant documents, or k is too small to capture any.
    return dcg / idcg


# --- Synthetic Data for Demonstration ---

# Query 1: "Latest advancements in quantum computing"
query1_gt = {'doc_qc_001', 'doc_qc_003', 'doc_qc_005', 'doc_qc_007'}
query1_retrieved = ['doc_qc_001', 'doc_ai_002', 'doc_qc_003', 'doc_ml_001', 'doc_qc_005', 'doc_robot_004']

# Query 2: "Impact of AI on healthcare ethics"
query2_gt = {'doc_ai_001', 'doc_ethics_003', 'doc_ai_005'}
query2_retrieved = ['doc_ai_001', 'doc_robot_001', 'doc_ethics_003', 'doc_ai_002', 'doc_ml_005']

# Query 3: "Sustainable energy solutions for urban areas"
query3_gt = {'doc_energy_001', 'doc_urban_002', 'doc_energy_003', 'doc_urban_005'}
query3_retrieved = ['doc_energy_001', 'doc_urban_002', 'doc_climate_001', 'doc_energy_003', 'doc_urban_001', 'doc_energy_005']


queries = [
    {'id': 'Q1', 'ground_truth': query1_gt, 'retrieved': query1_retrieved},
    {'id': 'Q2', 'ground_truth': query2_gt, 'retrieved': query2_retrieved},
    {'id': 'Q3', 'ground_truth': query3_gt, 'retrieved': query3_retrieved}
]

# --- Evaluate Metrics ---
k_value = 5 # Evaluate top 5 retrieved documents

all_recall_at_k = []
all_mrr = []
all_ndcg_at_k = []

print(f"Evaluating retrieval for k={k_value}:\n")

for query in queries:
    query_id = query['id']
    gt = query['ground_truth']
    ret = query['retrieved']

    recall = calculate_recall_at_k(gt, ret, k_value)
    mrr = calculate_mrr(gt, ret)
    ndcg = calculate_ndcg_at_k(gt, ret, k_value)

    all_recall_at_k.append(recall)
    all_mrr.append(mrr)
    all_ndcg_at_k.append(ndcg)

    print(f"Query {query_id}:")
    print(f"  Recall@{k_value}: {recall:.4f}")
    print(f"  MRR: {mrr:.4f}")
    print(f"  NDCG@{k_value}: {ndcg:.4f}")
    print("-" * 20)

print("\n--- Aggregate Results ---")
print(f"Average Recall@{k_value}: {np.mean(all_recall_at_k):.4f}")
print(f"Average MRR: {np.mean(all_mrr):.4f}")
print(f"Average NDCG@{k_value}: {np.mean(all_ndcg_at_k):.4f}")


### Interpreting the Output and Practical Use Cases

The code above demonstrates how to calculate Recall@k, MRR, and NDCG@k for a set of synthetic queries. Let's break down what the output means and how these metrics are applied in real-world RAG development.

#### Interpreting the Output

*   **Recall@k**: A score closer to 1.0 indicates that a high percentage of the truly relevant documents were found within the top `k` results. For example, a Recall@5 of 0.75 means that 75% of all known relevant documents for that query were present in the top 5 retrieved items. In RAG, high recall is often desired to ensure the LLM has access to a broad range of relevant information, reducing the risk of hallucination due to missing context.

*   **MRR**: A higher MRR (closer to 1.0) signifies that the first relevant document was found very early in the ranked list. An MRR of 0.5 means, on average, the first relevant document appeared at rank 2 (since 1/2 = 0.5). This metric is critical for systems where the user (or LLM) primarily cares about the very first relevant piece of information, such as a quick fact lookup or a direct answer. If your RAG system is designed for precise, single-answer queries, optimizing for MRR might be a priority.

*   **NDCG@k**: This metric provides a nuanced view, considering both the relevance of documents and their position. A higher NDCG@k (closer to 1.0) means that highly relevant documents are ranked higher in the list, and less relevant ones are pushed down. Unlike Recall, NDCG accounts for graded relevance (even if simplified to binary in our example). It's a robust metric for scenarios where the quality of the *entire* top-k list matters, and not just the presence of *any* relevant item. For RAG, a high NDCG suggests that the most pertinent information is readily available at the top of the context provided to the LLM, potentially leading to more focused and accurate generations.

#### Performance Trade-offs and Use Cases

Choosing which metric to prioritize depends heavily on the specific goals of your RAG system:

1.  **Maximizing Context Coverage (High Recall)**:
    *   **Scenario**: You're building a RAG system for complex research questions where the LLM needs to synthesize information from many sources. Missing a single relevant document could lead to an incomplete or incorrect answer.
    *   **Trade-off**: Achieving very high recall might sometimes come at the cost of precision (retrieving many irrelevant documents alongside relevant ones). This could bloat the LLM's context window, potentially leading to 'lost in the middle' phenomenon or increased inference costs.
    *   **Tuning**: Experiment with different `k` values. A larger `k` generally increases recall but also the context size.

2.  **Finding the Best Answer Quickly (High MRR)**:
    *   **Scenario**: Your RAG system is designed for quick Q&A, where users expect a direct, concise answer, and the first retrieved document is often sufficient. Think of a chatbot answering factual questions.
    *   **Trade-off**: Focusing solely on MRR might mean that while the *first* relevant document is excellent, other equally or more relevant documents might be ranked much lower or not retrieved at all.
    *   **Tuning**: Optimize your retriever's scoring function to prioritize the most confident, single-best match.

3.  **Optimizing Ranked List Quality (High NDCG)**:
    *   **Scenario**: You're building a RAG system for document summarization, report generation, or complex analytical tasks where the LLM needs to process a well-ordered set of documents with varying degrees of importance. This is often the most comprehensive metric for general-purpose RAG.
    *   **Trade-off**: NDCG is more complex to calculate and requires graded relevance judgments, which can be labor-intensive to obtain for ground truth data.
    *   **Tuning**: Focus on improving the ranking capabilities of your retriever, perhaps by using re-ranking models or more sophisticated similarity functions.

#### 2026 Readiness: Automated Evaluation and MLOps for RAG

In 2026, these metrics are not just for one-off evaluations. They are integral to automated RAG evaluation pipelines within MLOps frameworks. Teams are leveraging tools like `LangChain`, `LlamaIndex`, `Ragas`, or custom evaluation harnesses to:

*   **Continuous Integration/Continuous Deployment (CI/CD)**: Automatically run retrieval evaluations on new code commits or data updates to prevent regressions.
*   **A/B Testing**: Compare different retrieval algorithms (e.g., dense vs. sparse, different embedding models) in production or staging environments.
*   **Monitoring**: Track retrieval performance over time to detect data drift or performance degradation.
*   **Hyperparameter Tuning**: Systematically optimize retriever parameters (e.g., chunk size, top-k value, re-ranking thresholds) based on these metrics.

By understanding and applying Recall, MRR, and NDCG, developers can build more robust, accurate, and performant RAG systems that truly augment LLM capabilities.


### Resources

*   **Information Retrieval Metrics Explained**: A good overview of common IR metrics, including the ones discussed: [https://www.cs.cornell.edu/courses/cs4300/2019sp/lectures/lec15-eval.pdf](https://www.cs.cornell.edu/courses/cs4300/2019sp/lectures/lec15-eval.pdf)
*   **NDCG Wikipedia**: Detailed explanation of NDCG, including its mathematical formulation and variations: [https://en.wikipedia.org/wiki/Discounted_cumulative_gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)
*   **Ragas Documentation**: An open-source framework for RAG evaluation, which includes implementations of these and other RAG-specific metrics: [https://docs.ragas.io/en/latest/](https://docs.ragas.io/en/latest/)
*   **Hugging Face `evaluate` Library**: While not specific to RAG, this library provides a unified API for various evaluation metrics, which can be adapted for retrieval tasks: [https://huggingface.co/docs/evaluate/index](https://huggingface.co/docs/evaluate/index)
*   **LlamaIndex Evaluation Module**: Explore how LlamaIndex integrates evaluation into RAG pipelines: [https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html)
